In [ ]:
%%capture
# see comments in README on changes to the conda venv
import os
from pathlib import Path

# import modin.pandas as pd
import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from PIL import Image
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import UNDEFINED, DM_ALONE, HTN_ALONE, HIV_ALONE, HTN_DM
from intecomm_analytics.utils import get_primary_cohorts_by_categorical_column, get_primary_cohorts_for_continuous_var
from intecomm_rando.constants import COMMUNITY_ARM, FACILITY_ARM
from intecomm_analytics.utils import get_great_table, get_bp, get_vl, get_glucose, get_composite
from edc_constants.constants import NOT_APPLICABLE

# import modin.config as modin_cfg
# modin_cfg.Engine.put("ray")

In [ ]:
df_main = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df_main[(df_main.bmi >= 30.0) & (df_main.primary_cohort_str=="HIV_ALONE") ].gender.value_counts()

In [ ]:
from intecomm_analytics.dataframes import get_appt_df
df_appt = get_appt_df()

In [ ]:
df_main.groupby("primary_cohort_str").size()

In [ ]:
# df_main.query("hiv==1.0 and (htn==1.0 or dm==1.0)")
# df_main.query("hiv==0.0 and (htn==1.0 or dm==1.0)")


In [ ]:
df_main.query("site_id>=200").group_identifier.nunique()

In [ ]:
df_main.query("primary_gl_endline.isna() and primary_gl_controlled_endline.notna()")[['primary_gl_endline', 'primary_gl_controlled_endline', 'glucose_resulted_endline', 'glucose_value_endline', 'glucose_fasting_hours_endline']]

In [ ]:
# primary_cohort
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "primary_cohort")
dftbl = pd.DataFrame(tbl_dct)
mapping = {DM_ALONE:"Diabetes alone", HTN_ALONE:"Hypertension alone", HTN_DM:"Diabetes and hypertension", HIV_ALONE:"HIV alone", UNDEFINED:"UNDEFINED", "n":"n"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl = dftbl[dftbl["Statistics"]!="UNDEFINED"]
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Diabetes alone", "Hypertension alone", "Diabetes and hypertension", "HIV alone"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True).replace("0 (0.0%)", "NA")
dfnum = dftbl.iloc[0:1]
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
df_primary_cohort = dftbl.copy()

In [ ]:
tbl_dct

In [ ]:
# country
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "country")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", "TZ": "Tanzania", "UG": "Uganda"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Tanzania", "Uganda"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfcountry = dftbl.copy()

In [ ]:
dfcountry

In [ ]:
# gender
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "gender")
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Female", "Male"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfgender = dftbl.copy()
dfgender

In [ ]:

# years since diagnosis
tbl_dct = get_primary_cohorts_for_continuous_var(df_main, "years_since_dx", ["median_iqr", "mean"])
dftbl = pd.DataFrame(tbl_dct)
dftbl["Statistics"] = dftbl["Statistics"].map({"count": "n", "mean": "Mean, SD", "median_iqr": "Median, IQR"})
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, IQR", "Mean, SD"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfyearsdx= dftbl.copy()

In [ ]:
# education
df1 = df_main.copy()
df1["education"] = df1["education"].apply(lambda x: "missing" if pd.isna(x) else x)
mapping = {
    "n": "n",
    "no_formal_education": "no_formal_education",
    "primary": "primary",
    "secondary": "secondary_or_tertiary",
    "post_secondary": "secondary_or_tertiary",
    "tertiary": "secondary_or_tertiary",
    "missing": "missing"}
df1["education"] = df1["education"].map(mapping)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "education")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "no_formal_education": "No formal education",
    "primary": "Primary",
    "secondary_or_tertiary": "Secondary or tertiary",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "No formal education", "Primary", "Secondary or tertiary", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfed = dftbl.copy()

In [ ]:
# age_in_years
df1 = df_main.copy()
bins = [0,34, 49, 110]
labels = ["<35", "35-49", ">=50"]
df1.loc[df1.age_in_years.isna(), "bins"] = "Missing"
df1["bins"] = pd.cut(df1[df1.age_in_years.notna()]["age_in_years"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Mean, SD"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        data.append(
            f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['age_in_years'].mean(),1)} "
            f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['age_in_years'].std(),1)})"
        )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Mean, SD", "<35", "35-49", ">=50", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfage= dftbl.copy()

In [ ]:
# bmi
df1 = df_main.copy()
bins = [0.0,24.0, 30.0, 1000.0]
labels = ["<25.0", "25.0-29.9", ">=30.0"]
df1.loc[df1.bmi.isna(), "bins"] = "Missing"
df1.loc[df1.bmi.notna(), "bins"] = pd.cut(df1[df1.bmi.notna()]["bmi"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Mean, SD"]

df1["ncd"] = df1.ncd.astype("Float64")
df1["hiv_only"] = df1.hiv_only.astype("Float64")
df1["bmi"] = df1.bmi.astype("Float64")

for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        data.append(
            f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bmi'].mean(),1)} "
            f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bmi'].std(),1)})"
        )
dftbl.loc[len(dftbl)] = data

dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Mean, SD", *labels, "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfbmi= dftbl.copy()


In [ ]:
# smoker
df1 = df_main.copy()
df1["smoking_status"] = df1["smoking_status"].apply(lambda x: "missing" if pd.isna(x) else x)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "smoking_status")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "nonsmoker": "Non-smoker",
    "former_smoker": "Former smoker",
    "smoker": "Smoker",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Non-smoker", "Former smoker", "Smoker", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfsmoke = dftbl.copy()

In [ ]:
# alcohol_consumption
df1 = df_main.copy()
df1["alcohol_consumption"] = df1["alcohol_consumption"].apply(lambda x: "missing" if pd.isna(x) else x)
df1["alcohol_consumption"] = df1["alcohol_consumption"].apply(lambda x: "never" if x==NOT_APPLICABLE else x)
df1.alcohol_consumption.value_counts(dropna=False)

tbl_dct = get_primary_cohorts_by_categorical_column(df1, "alcohol_consumption")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "never": "Never",
    "occasionally": "Occasionally",
    "1_2_per_week": "1-2 times per week",
    "3_4_per_week": "3-4 times per week",
    "daily": "Daily",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=[*mapping.values()], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfalcohol = dftbl.copy()

In [ ]:
# marital status
df1 = df_main.copy()
df1["marital_status"] = df1["marital_status"].apply(lambda x: "missing" if pd.isna(x) else x)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "marital_status")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "married": "Married",
    "widowed": "Widowed",
    "divorced": "Divorced",
    "single":"Single",
    "missing": "Missing"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Single", "Married", "Divorced", "Widowed", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "marital_status"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfmarital = dftbl.copy()

In [ ]:
# bp_controlled_baseline
df1 = df_main.copy()
cond = ((df1.primary_cohort==HTN_ALONE) | (df1.primary_cohort==HTN_DM))
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension"
dftbl= get_bp(df1, "bp_controlled_baseline", cond, label)
df_htn_bp_base = dftbl.copy()

In [ ]:
# bp_controlled_baseline (alone)
df1 = df_main.copy()
cond = (df1.primary_cohort==HTN_ALONE)
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension alone"
dftbl= get_bp(df1, "bp_controlled_baseline", cond, label)
df_htn_alone_bp_base = dftbl.copy()

In [ ]:
# glucose_controlled_baseline alone
df1 = df_main.copy()
cond = (df1.primary_cohort==DM_ALONE)
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes alone"
dftbl= get_glucose(df1, "glucose_controlled_baseline", cond, label)
df_dm_alone_glu_base = dftbl.copy()

In [ ]:
# glucose_controlled_baseline
df1 = df_main.copy()
cond = ((df1.primary_cohort==DM_ALONE) | (df1.primary_cohort==HTN_DM))
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes"
dftbl= get_glucose(df1, "glucose_controlled_baseline", cond, label)
df_dm_glu_base = dftbl.copy()

In [ ]:
# vl_controlled_baseline
df1 = df_main.copy()
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<1000 copies per mL"
dftbl = get_vl(df1, "vl_controlled_baseline", cond, label)
dfvl_base = dftbl.copy()

In [ ]:
# vl_controlled_baseline_400
df1 = df_main.copy()
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<400 copies per mL"
dftbl = get_vl(df1, "vl_controlled_baseline_400", cond, label)
dfvl400_base = dftbl.copy()

In [ ]:
# vl_controlled_baseline_50
df1 = df_main.copy()
cond = ((df1.hiv==1) & (df1.dm==0) & (df1.htn==0))
label = "<50 copies per mL"
dftbl = get_vl(df1, "vl_controlled_baseline_50", cond, label)
dfvl50_base = dftbl.copy()

In [ ]:
df1 = df_main.copy()
cond = ((df1.primary_cohort==DM_ALONE) | (df1.primary_cohort==HTN_DM) | (df1.primary_cohort==HTN_ALONE))
label = "Controlled"
dftbl = get_composite(df1, "primary_composite_baseline", cond, label)
dfcomposite_base = dftbl.copy()


In [ ]:
dfvlall_base = pd.concat([dfvl_base, dfvl400_base, dfvl50_base])
dfvlall_base = dfvlall_base.reset_index(drop=True)

In [ ]:
groupings = [
    (dfnum, [""]),
    (dfcountry, ["Site"]),
    (df_primary_cohort, ["Condition"]),
    (dfyearsdx, ["Years living with condition"]),
    (dfgender, ["Sex"]),
    (dfage, ["Age, years"]),
    (dfbmi, ["BMI, kg/m2"]),
    (dfmarital, ["Marital status"]),
    (dfed, ["Education"]),
    (dfsmoke, ["Smoking"]),
    (dfalcohol, ["Alcohol consumption"]),
]
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)
# convert to GT
baseline_table = get_great_table(dftbl_final, group_row_headers, "Table 1: Baseline characteristics")
baseline_table.show()

In [ ]:
# save as png
baseline_table.save(analysis_folder / "baseline_characteristics.png")
# export to PDF
image = Image.open(analysis_folder / "baseline_characteristics.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "baseline_characteristics.pdf", "PDF", resolution=800, optimize=True, quality=95)

In [ ]:
groupings = [
    (dfnum, [""]),
    (dfcountry, ["Site"]),
    (dfvlall_base, ["HIV viral load"]),
    (df_htn_alone_bp_base, ["Blood pressure baseline"]),
    (df_htn_bp_base, ["Blood pressure baseline"]),
    (df_dm_alone_glu_base, ["Fasting blood glucose baseline (fasted 8hrs+)"]),
    (df_dm_glu_base, ["Fasting blood glucose baseline (fasted 8hrs+)"]),
    (dfcomposite_base, ["Composite controlled baseline (BP/Glucose)"])
]
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)
# convert to GT
baseline_table = get_great_table(dftbl_final, group_row_headers, "Table 1.1: Baseline clinical measures")
baseline_table.show()

In [ ]:
# save as png
baseline_table.save(analysis_folder / "baseline_clinical_measures.png")
# export to PDF
image = Image.open(analysis_folder / "baseline_clinical_measures.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "baseline_clinical_measures.pdf", "PDF", resolution=800, optimize=True, quality=95)

In [ ]:
df_main

In [ ]:
df_main2 = df_main[df_main.retained_6m==1].copy()


In [ ]:
df1 = df_main2.copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["onstudy_bins"] = pd.cut(df1["onstudy_days"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "onstudy_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Mean, SD"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        data.append(
            f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['onstudy_days'].mean(),1)} "
            f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['onstudy_days'].std(),1)})"
        )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Mean, SD", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfonstudy= dftbl.copy()


In [ ]:
df1 = df_main2.copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["death_days_to_event_bins"] = pd.cut(df1["death_days_to_event"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "death_days_to_event_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Median, (min-max)"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        if df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['death_days_to_event'].isna().all():
            data.append("0 (0-0)")
        else:
            data.append(
                f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['death_days_to_event'].median(),1)} "
                f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['death_days_to_event'].min(),1)}-{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['death_days_to_event'].max(),1)})"
            )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, (min-max)", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (nan%)", "0 (0.0%)")
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfdeathdays= dftbl.copy()



In [ ]:
df1 = df_main2[df_main2.primary_cohort.isin([HIV_ALONE])].copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["vl_days_to_event_bins"] = pd.cut(df1["vl_days_to_event"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "vl_days_to_event_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Median, (min-max)"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        if df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].isna().all():
            data.append("0 (0-0)")
        else:
            data.append(
                f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].median(),1)} "
                f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].min(),1)}-{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].max(),1)})"
            )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, (min-max)", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (nan%)", "0 (0.0%)")
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfdeathdays= dftbl.copy()



In [ ]:
df1 = df_main2[df_main2.primary_cohort.isin([HTN_ALONE, HTN_DM])].copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["bp_days_to_event_bins"] = pd.cut(df1["bp_days_to_event"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "bp_days_to_event_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Median, (min-max)"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        if df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].isna().all():
            data.append("0 (0-0)")
        else:
            data.append(
                f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].median(),1)} "
                f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].min(),1)}-{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].max(),1)})"
            )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, (min-max)", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (nan%)", "0 (0.0%)")
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfbpdays= dftbl.copy()

In [ ]:
df1 = df_main2[df_main2.primary_cohort.isin([DM_ALONE, HTN_DM])].copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["glucose_days_to_event_bins"] = pd.cut(df1["glucose_days_to_event"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "glucose_days_to_event_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Median, (min-max)"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        if df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['glucose_days_to_event'].isna().all():
            data.append("0 (0-0)")
        else:
            data.append(
                f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['glucose_days_to_event'].median(),1)} "
                f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['glucose_days_to_event'].min(),1)}-{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['glucose_days_to_event'].max(),1)})"
            )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, (min-max)", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (nan%)", "0 (0.0%)")
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfgludays= dftbl.copy()


In [ ]:
groupings = [
    (dfnum, [""]),
    (dfcountry, ["Site"]),
    (dfbpdays, ["Blood pressure"]),
    (dfgludays, ["Fasting blood glucose"]),
    (dfgludays, ["HIV viral load"]),
]
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)
# convert to GT
outcomes_table = get_great_table(dftbl_final, group_row_headers, "Table 1.3: Days to clinical outcomes")
outcomes_table.show()

In [ ]:
# save as png
outcomes_table.save(analysis_folder / "days_to_clinical_outcomes.png")
# export to PDF
image = Image.open(analysis_folder / "days_to_clinical_outcomes.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "days_to_clinical_outcomes.pdf", "PDF", resolution=800, optimize=True, quality=95)

In [ ]:
from intecomm_analytics.dataframes.glucose import get_all_glucose_results

df_glu = get_all_glucose_results(df_main)

In [ ]:
len(df_glu.subject_identifier.unique())

In [ ]:
len(df_glu[df_glu.glucose_date_delta.dt.days<60].subject_identifier.unique())

In [ ]:
len(df_main[(df_main.dm==1) & (df_main.hiv==0)])

In [ ]:
df_glu[["baseline_datetime", "glucose_date", "glucose_date_delta"]]

In [ ]:
df_missing = df_main[df_main.glucose_value_baseline.isna()][[                "subject_identifier",
                "glucose_date_baseline",
                "glucose_value_baseline",
                "dm","primary_cohort","primary_cohort_str", "assignment", "country"
]].copy().reset_index(drop=True)

In [ ]:
df_missing.primary_cohort_str.value_counts()

In [ ]:
df_glu = df_glu.merge(df_main[["subject_identifier", "dm","primary_cohort","primary_cohort_str", "assignment", "country"]], on="subject_identifier", how="left")

In [ ]:
df_missing = df_glu[df_glu.glucose_value.isna()][[
    "subject_identifier",
                "baseline_datetime", "glucose_date", "glucose_date_delta", "glucose_value",
                "dm","primary_cohort","primary_cohort_str", "assignment", "country"
]].copy().reset_index(drop=True)

In [ ]:
df_missing

In [ ]:
df_main[[col for col in list(df_main.columns) if col.startswith("primary_")]]

In [ ]:
import statsmodels.api as sm


In [ ]:
df_model = df_main[["primary_cohort_str", "subject_identifier", "group_identifier", "primary_bp_controlled_endline", "assignment"]].copy()
df_model["primary_bp_controlled_endline"] = df_model["primary_bp_controlled_endline"].fillna(0)
df_model = df_model[df_model["primary_cohort_str"].isin([ "HTN_ALONE"])]
df_model

In [ ]:
model = sm.GEE.from_formula(
    'primary_bp_controlled_endline ~ assignment',
    groups='subject_identifier',
    data=df_model,
    family=sm.families.Binomial(link=sm.families.links.Identity()),
    cov_struct=sm.cov_struct.Exchangeable()
)

In [ ]:
result = model.fit()
print(result.summary())

In [ ]:
model = sm.GEE.from_formula(
    'retained_6m ~ assignment',
    groups='group_identifier',
    data=df_main[["group_identifier", "retained_6m", "assignment"]],
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

In [ ]:
result = model.fit()
print(result.summary())